In [ ]:
import logging
import matplotlib.pyplot as plt
import pandas as pd
import torch
from genkit.datasets import fetch_synthetic_data
from genkit.diffusion import DLPMEps
from genkit.metrics import mmd_rbf, tail_coverage_error
from genkit.nn import MLPModel
from genkit.training import train


In [ ]:
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", force=True)
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print(f"[INFO] device={device} dtype={dtype}")


In [ ]:
class ZeroNet(torch.nn.Module):
    def forward(self, x, t):
        return torch.zeros_like(x)


class DLPMMonteCarloScoreNet(torch.nn.Module):
    def __init__(self, model, n_mc=2048, chunk_size=512):
        super().__init__()
        self.model = model
        self._n_mc = int(n_mc)
        self._chunk_size = int(chunk_size)
        if self._n_mc <= 0:
            raise ValueError("n_mc must be positive.")
        if self._chunk_size <= 0:
            raise ValueError("chunk_size must be positive.")

    @torch.no_grad()
    def forward(self, x, t):
        x = x.to(device=self.model._device, dtype=self.model._fdtype)
        t_idx = (t.reshape(-1) * self.model._n_steps).round().long()
        t_idx = t_idx.clamp(1, self.model._n_steps - 1)
        out = torch.empty_like(x)
        tiny = torch.finfo(x.dtype).tiny

        for start in range(0, x.shape[0], self._chunk_size):
            stop = min(start + self._chunk_size, x.shape[0])
            x_chunk = x[start:stop]
            x_flat = x_chunk.reshape(x_chunk.shape[0], -1)
            x2 = x_flat.square().sum(dim=1, keepdim=True)

            A = self.model._draw_A(self._n_mc).reshape(1, -1).clamp_min(tiny)
            logw = -0.5 * (self.model._dim * A.log() + x2 / A)
            weights = torch.softmax(logw, dim=1)
            score = -x_flat * (weights / A).sum(dim=1, keepdim=True)

            sigma = self.model._sigma_1_t.index_select(0, t_idx[start:stop]).reshape(-1, 1)
            eps_hat = -sigma * score
            out[start:stop] = eps_hat.reshape_as(x_chunk)

        return out


In [ ]:
alpha = 1.7
print(f"[INFO] loading isotropic alpha-stable data alpha={alpha}")
x_train, _, x_test = fetch_synthetic_data(
    "alpha_stable",
    alpha=alpha,
    dim=2,
    n_samples=170_000,
    val_size=0.001,
    test_size=0.8,
    standardize=False,
    device=device,
    dtype=dtype,
)
n_train = x_train.shape[0]
n_test = x_test.shape[0]
print(f"[INFO] train_shape={n_train} test_shape={n_test}")


In [ ]:
n_steps = 64
mc_score_samples = 4096
mc_chunk_size = 1024


def make_net():
    return MLPModel(dim=2, width=128, depth=3).to(device=device, dtype=dtype)


def make_models(seed):
    torch.manual_seed(seed)
    print(f"[INFO] building models seed={seed}")
    dlpm = DLPMEps(
        net=make_net(),
        dim=2,
        n_steps=n_steps,
        alpha=alpha,
        n_trial_A=1,
        n_trial_G=1,
        reduce_type="mean",
        device=device,
    )
    mc_dlpm = DLPMEps(
        net=ZeroNet(),
        dim=2,
        n_steps=n_steps,
        alpha=alpha,
        n_trial_A=1,
        n_trial_G=1,
        reduce_type="mean",
        device=device,
    )
    mc_dlpm._net = DLPMMonteCarloScoreNet(
        mc_dlpm,
        n_mc=mc_score_samples,
        chunk_size=mc_chunk_size,
    ).to(device=device, dtype=dtype)
    return {"DLPM": dlpm, "DLPM + MC score": mc_dlpm}


tce_probs = {
    "TCE(90)": 1e-1,
    "TCE(95)": 5e-2,
    "TCE(99)": 1e-2,
    "TCE(99,9)": 1e-3,
    "TCE(99,99)": 1e-4,
}


@torch.no_grad()
def evaluate_model(name, model, trial, x_ref, n_tail=25_000, n_mmd=5_000):
    print(f"[INFO] evaluating trial={trial} model={name} n_tail={n_tail} n_mmd={n_mmd}")
    x_ref = x_ref[:n_tail].detach().cpu().to(torch.float64)
    x_gen = model.sample(len(x_ref)).detach().cpu().to(torch.float64)
    row = {
        "trial": trial,
        "model": name,
        "mmd_rbf": mmd_rbf(x_ref[:n_mmd], x_gen[:n_mmd]),
    }
    for label, prob in tce_probs.items():
        probs = torch.tensor([prob], dtype=torch.float64)
        row[label] = tail_coverage_error(x_ref, x_gen, probs=probs, tail="upper", mode="log")
    return row


In [ ]:
n_tail = n_test
n_mmd = n_test // 10
n_trials = 5
train_kwargs = dict(batch_size=128, n_epochs=32, lr=5e-4, device=device, use_adamw=False, lr_schedule="constant", freq_logging=8)

In [ ]:
print(f"[INFO] n_trials={n_trials} train_kwargs={train_kwargs}")
print(f"[INFO] mc_score_samples={mc_score_samples} mc_chunk_size={mc_chunk_size}")

rows = []
for trial in range(1, n_trials + 1):

    print(f"[INFO] trial {trial}/{n_trials}: start")
    models = make_models(seed=trial - 1)

    for name, model in models.items():

        if name == "DLPM":
            print(f"[INFO] trial {trial}/{n_trials}: train {name}")
            model, _ = train(model, x_train, **train_kwargs)
        else:
            print(f"[INFO] trial {trial}/{n_trials}: skip training for {name}")

        print(f"[INFO] trial {trial}/{n_trials}: evaluate {name}")
        rows.append(evaluate_model(name, model, trial, x_test, n_tail=n_tail, n_mmd=n_mmd))

    print(f"[INFO] trial {trial}/{n_trials}: done")


In [ ]:
print("[INFO] computing test-vs-test reference metrics")

x_ref = x_test.detach().cpu().to(torch.float64)

for trial in range(1, n_trials + 1):
    generator = torch.Generator(device="cpu").manual_seed(10_000 + trial)
    perm = torch.randperm(x_ref.shape[0], generator=generator)

    x_a = x_ref[perm[:n_tail]]
    x_b = x_ref[perm[n_tail: 2 * n_tail]]

    row = {
        "trial": trial,
        "model": "test vs test",
        "mmd_rbf": mmd_rbf(x_a[:n_mmd], x_b[:n_mmd]),
    }

    for label, prob in tce_probs.items():
        probs = torch.tensor([prob], dtype=torch.float64)
        row[label] = tail_coverage_error(x_a, x_b, probs=probs, tail="upper", mode="log")

    rows.append(row)

In [ ]:
print("[INFO] aggregating results")
results = pd.DataFrame(rows)
tce_cols = ["TCE(90)", "TCE(99)", "TCE(99,9)", "TCE(99,99)"]
tce_x = [0.90, 0.99, 0.999, 0.9999]

grouped = results.groupby("model")
mean_results = grouped[["mmd_rbf", *tce_cols]].mean()
std_results = grouped[tce_cols].std().fillna(0.0)

fig, ax = plt.subplots(figsize=(6, 5))

for model_name, row in mean_results.iterrows():
    y = row[tce_cols].to_numpy(dtype=float)
    yerr = std_results.loc[model_name, tce_cols].to_numpy(dtype=float)
    lower = (y - yerr).clip(min=1e-12)
    upper = y + yerr
    label = f"{model_name} (MMD-RBF={row['mmd_rbf']:.4g})"
    line, = ax.plot(tce_x, y, marker="o", linewidth=2, label=label, alpha=0.5)
    ax.fill_between(tce_x, lower, upper, color=line.get_color(), alpha=0.2)

ax.set_xscale("logit")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-2)
ax.set_xticks(tce_x)
ax.set_xticklabels(["90", "99", "99.9", "99.99"])
ax.set_xlabel("tail quantile (%)", fontsize=12)
ax.set_ylabel("TCE, upper tail log error", fontsize=12)

ax.grid(True, which="both", alpha=0.4)
ax.legend()

plt.tight_layout()
